# Прогноз заездов в городском отеле
**Этап 2 · EDA, baseline и поиск аномалий**

Пока работаем с [H2](https://pmc.ncbi.nlm.nih.gov/articles/PMC6297060/) — данными городского отеля в Лиссабоне с плановыми заездами с июля 2015 по август 2017 года. Параллельно мы написали Нуно Антониу, автору и куратору набора [Hotel bookings data 2014–2023](https://zenodo.org/records/18119072), и запросили обезличенные записи по городским отелям C1 и C2. Для проекта подойдёт выгрузка хотя бы по одному отелю. Мы попросили включить плановую дату заезда, дату бронирования или LeadTime, итоговый статус и дату его присвоения, а также канал бронирования.

После ответа и получения доступа сверим определения полей и проверим, пересекаются ли записи с H2. Затем приведём данные к общей схеме и объединим их, сохранив признак отеля. Это позволит увеличить выборку и расширить временное окно анализа.

Для каждой даты заезда прогнозируем число заездов из броней, активных за 14, 7 или 3 дня до этой даты. Одна запись о заезде — одна бронь со статусом Check-Out. Считаем baseline по исторической доле заездов и проверяем, как на дневной прогноз влияют отдельные и объединённые флаги.

Для запуска откройте ноутбук из корня репозитория в окружении с зависимостями из `requirements.txt`. Оригинальный `H2.csv` можно положить в `data/`. Если локальной копии нет, ноутбук скачает данные и сверит SHA-256.

In [ ]:
import calendar
import hashlib
import io
import zipfile
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from IPython.display import Image, Markdown, display
from sklearn.metrics import mean_absolute_error

from src.forecast import (
    detect_anomalies, forecast_baseline, make_snapshot, temporal_split,
)

SOURCE = 'https://ars.els-cdn.com/content/image/1-s2.0-S2352340918315191-mmc2.zip'
MIRROR = 'https://raw.githubusercontent.com/afaq-ahmad/Hotel-booking-demand-Case-Study/master/H2.csv'
H2_SHA256 = '5107090ffdc5585a9e18299801e9d85c3b459c235542882183a502a73ef63c4d'
HORIZONS = (14, 7, 3)
SEED = 42

def load_h2(path=Path('data/H2.csv')):
    """Проверяет, что загружен именно оригинальный H2."""
    if path.is_file():
        payload = path.read_bytes()
        source = path.name
    else:
        try:
            response = requests.get(SOURCE, timeout=30)
            response.raise_for_status()
            with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
                payload = archive.read('H2.csv')
            source = 'Приложение издателя mmc2.zip'
        except (requests.RequestException, zipfile.BadZipFile, KeyError):
            response = requests.get(MIRROR, timeout=45)
            response.raise_for_status()
            payload = response.content
            source = 'Зеркало H2.csv с проверкой SHA-256'
    payload = payload.replace(b'\r\n', b'\n')
    if hashlib.sha256(payload).hexdigest() != H2_SHA256:
        raise ValueError('Содержимое H2.csv отличается от версии издателя.')
    return pd.read_csv(
        io.BytesIO(payload), keep_default_na=False, na_values=['NA', ''],
        dtype={'Agent': 'string', 'Company': 'string', 'Country': 'string'}
    ), source

def show_figure(fig):
    output = io.BytesIO()
    fig.savefig(output, format='png', dpi=135, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    display(Image(data=output.getvalue()))

plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'axes.axisbelow': True,
                     'grid.alpha': 0.22, 'figure.dpi': 110})
raw, source = load_h2(Path('H2.csv') if Path('H2.csv').is_file() else Path('data/H2.csv'))
assert raw.shape == (79_330, 31)
print(f'{source}. Размер H2: {len(raw):,} строк × {raw.shape[1]} столбец.')


## 1. Первичный обзор H2

Сначала выведем первые пять строк оригинального CSV, затем проверим типы полей, явные пропуски и значения NULL.


In [ ]:
with pd.option_context('display.max_columns', None):
    display(raw.head())

overview = pd.DataFrame({
    'Тип': raw.dtypes.astype(str),
    'Заполнено': raw.notna().sum(),
    'Явных пропусков': raw.isna().sum(),
    'Уникальных значений без пропусков': raw.nunique()
})
overview['Строковый NULL'] = 0
for column in ['Country', 'Agent', 'Company']:
    overview.loc[column, 'Строковый NULL'] = raw[column].str.strip().eq('NULL').sum()
display(overview)


## 2. Даты и статус бронирования

Дата создания = плановая дата заезда минус LeadTime. Используем срез **на конец календарного дня**. Бронь входит в когорту, если к этой дате она создана и её финальная отмена ещё не произошла. No-Show остаётся в когорте до заезда и даёт ноль в целевой переменной. Снимки приближённые: CSV содержит одну запись на бронь и дату последнего статуса, поэтому история переносов и промежуточных статусов неизвестна. Дни без активных броней включаем в оценку с фактом и прогнозом 0.

Финальные статусы и их даты нужны только для восстановления когорты и разметки. Изменяемые поля вроде длительности, числа гостей и ADR используем для диагностики. Канал DistributionChannel нужен для ретроспективного среза качества. Авторы использовали значения из журнала накануне заезда, когда они были доступны, поэтому для прогноза на ранних горизонтах берём календарь, LeadTime и признаки, которые вычисляются из восстановленной активной когорты.


Срезы восстановлены ретроспективно по опубликованным датам. Для броней с переносом заезда в H2 нет цепочки предыдущих плановых дат. Поля гостей, ночей, тарифа и канала используем только в описательном анализе и ручной проверке. Часть полей исходной выгрузки зафиксирована за день до заезда. [Описание формирования H2](https://pmc.ncbi.nlm.nih.gov/articles/PMC6297060/#sec2).

In [ ]:
bookings = raw.copy()
months = {name: i for i, name in enumerate(calendar.month_name) if i}
bookings['arrival'] = pd.to_datetime({
    'year': bookings['ArrivalDateYear'],
    'month': bookings['ArrivalDateMonth'].map(months),
    'day': bookings['ArrivalDateDayOfMonth']
})
bookings['status_date'] = pd.to_datetime(bookings['ReservationStatusDate'], format='%Y-%m-%d')
bookings['created'] = bookings['arrival'] - pd.to_timedelta(bookings['LeadTime'], unit='D')
bookings['show'] = bookings['ReservationStatus'].eq('Check-Out').astype('int8')
bookings['canceled'] = bookings['ReservationStatus'].eq('Canceled')
bookings['nights'] = bookings['StaysInWeekendNights'] + bookings['StaysInWeekNights']
bookings['guests'] = bookings[['Adults', 'Children', 'Babies']].sum(axis=1, min_count=3)
bookings['ready'] = bookings[['arrival', 'status_date']].max(axis=1)
dates = pd.date_range(bookings['arrival'].min(), bookings['arrival'].max(), name='arrival')

assert bookings[['arrival', 'status_date', 'created']].notna().all().all()
assert bookings['ReservationStatus'].isin(['Check-Out', 'Canceled', 'No-Show']).all()
assert bookings['show'].eq(1 - bookings['IsCanceled']).all()
assert bookings['status_date'].ge(bookings['created']).all()
assert (bookings['status_date'].le(bookings['arrival']) | ~bookings['canceled']).all()
assert (bookings['status_date'].ge(bookings['arrival']) | bookings['canceled']).all()

status_counts = bookings['ReservationStatus'].value_counts()
audit = pd.Series({
    'Полные совпадения по 31 полю сверх первой строки': int(raw.duplicated().sum()),
    'Указано 0 гостей': int(bookings['guests'].eq(0).sum()),
    'Указано 0 ночей': int(bookings['nights'].eq(0).sum()),
    'Пропущено число детей': int(raw['Children'].isna().sum()),
    'Страна записана как NULL': int(raw['Country'].str.strip().eq('NULL').sum()),
    'Агент записан как NULL': int(raw['Agent'].str.strip().eq('NULL').sum()),
    'Компания записана как NULL': int(raw['Company'].str.strip().eq('NULL').sum())
}, name='Строк')
print(f'Заезды запланированы с {dates.min():%d.%m.%Y} по {dates.max():%d.%m.%Y} ({len(dates)} дат).')
display(status_counts.rename('Броней').to_frame())
display(audit.to_frame())
display(bookings.nlargest(5, 'ADR').round({'ADR': 2})[
    ['arrival', 'ADR', 'nights', 'guests', 'LeadTime', 'ReservationStatus']
])


Совпадающие строки сохраняем. В H2 нет ID брони, а совпадение полей совместимо с несколькими реальными бронями. По словарю статьи NULL в Agent и Company означает «не применимо». Нулевые ночи, нулевое число гостей и крайний ADR проверяем как кандидаты на ошибку или особый тип записи. Эти поля не влияют на вычисление когорты или прогноз. Для эксперимента с обработкой аномалий ниже используем признаки, которые доступны к дате среза.


## 3. Распределения и динамика бронирований

Посмотрим на числовые признаки и их хвосты, статус брони по месяцам, а также на объём и долю заездов по каналам. Здесь используем весь H2 для описания данных. Проверка гипотез о прогнозе ниже использует хронологическое разбиение.


In [ ]:
numeric = ['LeadTime', 'nights', 'guests', 'ADR', 'BookingChanges', 'DaysInWaitingList']
display(bookings[numeric].describe(percentiles=[.25, .5, .75, .95, .99]).round(2))

fig, axes = plt.subplots(2, 2, figsize=(11.6, 7.2), layout='constrained')
histograms = [
    ('LeadTime', np.arange(0, 651, 14), 'Срок бронирования', 'От бронирования до заезда, дни'),
    ('nights', np.arange(-.5, bookings['nights'].max() + 1.5), 'Длительность проживания', 'Ночей'),
    ('guests', np.arange(-.5, bookings['guests'].max() + 1.5), 'Размер брони', 'Гостей'),
]
for ax, (column, edges, title, label) in zip(axes.flat, histograms):
    ax.hist(bookings[column].dropna(), bins=edges, color='#277887', log=True)
    ax.set(title=title, xlabel=label, ylabel='Броней, лог. шкала')
adr_ax = axes[1, 1]
adr_cap = bookings['ADR'].quantile(.99)
adr_ax.hist(bookings.loc[bookings['ADR'].le(adr_cap), 'ADR'], bins=35, color='#c26d35')
adr_ax.set(title='ADR до 99-го процентиля', xlabel='Средняя стоимость номера за ночь',
           ylabel='Броней')
show_figure(fig)

monthly_input = bookings.assign(month=bookings['arrival'].dt.to_period('M'))
monthly = monthly_input.groupby(['month', 'ReservationStatus']).size().unstack(fill_value=0)
monthly.index = monthly.index.to_timestamp()
month_outcomes = [('Check-Out', 'Заезд', '#277887'), ('Canceled', 'Отмена', '#c26d35'),
                  ('No-Show', 'Неявка', '#76549b')]
raw_channels = bookings.groupby('DistributionChannel')['show'].agg(['size', 'mean'])
raw_channels = raw_channels.sort_values('size', ascending=False)
display(raw_channels.rename(columns={'size': 'Броней', 'mean': 'Доля заездов'}).round(3))

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4.1), layout='constrained',
                                  gridspec_kw={'width_ratios': [1.55, 1]})
for status, label, color in month_outcomes:
    left.plot(monthly.index, monthly[status], marker='o', markersize=2, label=label, color=color)
left.set(title='Итоги броней по месяцу планового заезда', xlabel='Месяц планового заезда',
         ylabel='Броней')
left.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
left.xaxis.set_major_formatter(mdates.DateFormatter('%m.%Y'))
left.legend(frameon=False)
channels = raw_channels.sort_values('size')
right.barh(channels.index, channels['size'], color='#277887')
right.set(title='Объём по каналам бронирования', xlabel='Броней, лог. шкала', xscale='log')
show_figure(fig)

print(f"Медиана LeadTime — {bookings['LeadTime'].median():.0f} дня, "
      f"99-й процентиль — {bookings['LeadTime'].quantile(.99):.0f}, "
      f"максимум — {bookings['LeadTime'].max()} дней.")
print(f"За пределами показанного диапазона ADR — "
      f"{bookings['ADR'].gt(adr_cap).sum()} броней, максимум — {bookings['ADR'].max():.0f}.")
print(f"На {raw_channels.index[0]} приходится {raw_channels['size'].iloc[0] / len(bookings):.1%} записей.")


## 4. Когорты на горизонтах 14, 7 и 3 дня

Сравним размеры когорт, долю заездов и чувствительность к выбору начала или конца календарного дня.


In [ ]:
snapshots = {h: make_snapshot(bookings, dates, h) for h in HORIZONS}
assert set(snapshots[14][0].index[snapshots[14][0]['show'].eq(1)]) <= set(
    snapshots[7][0].index[snapshots[7][0]['show'].eq(1)]
) <= set(snapshots[3][0].index[snapshots[3][0]['show'].eq(1)])

cohorts = pd.DataFrame([
    {'Горизонт, дни': h, 'Активные брони': len(active),
     'Заезды в когорте': int(active['show'].sum()),
     'Доля заездов, %': 100 * active['show'].mean(),
     'Даты без активных броней': daily['n'].eq(0).sum(),
     'Разница состава на начало и конец дня, броней': change}
    for h, (active, daily, change) in snapshots.items()
]).set_index('Горизонт, дни')
display(cohorts.round(2))


## 5. Временная валидация и EDA активных броней

Исторический период нужен для расчёта доли заездов, границ IQR и подгонки Isolation Forest. На следующем периоде смотрим устойчивость флагов. Финальный тест — с 1 мая по 31 августа 2017. На границах оставляем запас. Из исторического периода убираем даты, для которых итог хотя бы одной активной брони стал известен в день первого прогноза следующего периода или позже. Это учитывает и поздние выезды. Все варианты baseline оцениваем на одинаковых датах.


Основная метрика — MAE в заездах на дату. Она измеряет средний абсолютный промах в исходных единицах, одинаково взвешивает даты и допускает нулевой факт. Для выбора одной стратегии усредняем MAE по трём горизонтам с равными весами. В таблицах также показываем MAE по каждому горизонту.

In [ ]:
WINDOWS = {
    'train': ('2015-07-01', '2016-12-15'),
    'validation': ('2017-01-01', '2017-02-14'),
    'test': ('2017-05-01', '2017-08-31'),
}

splits = {h: temporal_split(d, h, WINDOWS) for h, (_, d, _) in snapshots.items()}
assert all(len(split['test']) == 123 for split in splits.values())
split_rows = []
for h, (active, daily, _) in snapshots.items():
    for part, day_index in splits[h].items():
        split_rows.append({'Горизонт': h, 'Период': part, 'Дат': len(day_index),
                           'Броней': daily.loc[day_index, 'n'].sum(),
                           'Первая дата': day_index.min().date().isoformat(),
                           'Последняя дата': day_index.max().date().isoformat()})
display(pd.DataFrame(split_rows).set_index(['Горизонт', 'Период']))

# Разведочные срезы на обучении. Горизонт 7 дней показан подробно как средний из трёх.
active7, daily7, _ = snapshots[7]
train7 = active7.loc[active7['arrival'].isin(splits[7]['train'])]
eda7 = daily7.loc[splits[7]['train']]
lead_labels = ['7–13', '14–29', '30–59', '60–119', '120+']
lead_group = pd.cut(train7['LeadTime'], bins=[7, 14, 30, 60, 120, np.inf],
                    labels=lead_labels, right=False)
lead_summary = train7.groupby(lead_group, observed=True)['show'].agg(['size', 'mean'])
channel_eda = train7.groupby('DistributionChannel')['show'].agg(['size', 'mean']).sort_values('size', ascending=False)
display(channel_eda.rename(columns={'size': 'Активных броней', 'mean': 'Доля заездов'}).round(3))

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 3.7), layout='constrained')
left.plot(eda7.index, eda7['n'].rolling(7, min_periods=1).mean(), label='Активные брони', color='#277887')
left.plot(eda7.index, eda7['actual'].rolling(7, min_periods=1).mean(), label='Заезды из них', color='#c26d35')
left.set(title='Когорта за 7 дней на обучении', ylabel='Число броней, среднее за 7 дат', xlabel='Дата планового заезда')
left.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
left.xaxis.set_major_formatter(mdates.DateFormatter('%m.%Y'))
left.legend(frameon=False)
bars = right.bar(lead_summary.index, lead_summary['mean'] * 100, color='#277887')
right.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
right.set(title='Доля заездов по сроку бронирования', xlabel='Дней от бронирования до заезда', ylabel='Заезды, %', ylim=(0, 100))
show_figure(fig)


weekdays = ['Пн', 'Вт', 'Ср', 'Чт', 'Пт', 'Сб', 'Вс']
weekday_eda = eda7.groupby(eda7.index.day_of_week).agg(
    Дат=('n', 'size'), Броней=('n', 'sum'), Заездов=('actual', 'sum'),
    Медиана_броней=('n', 'median'),
)
weekday_eda.index = weekdays
weekday_eda['Доля заездов, %'] = 100 * weekday_eda['Заездов'] / weekday_eda['Броней']
display(weekday_eda.round(2))

**Гипотезы.** Доля заездов меняется со сроком бронирования и днём недели. Проверим, даёт ли учёт этих признаков выигрыш у исторической средней. Редкое сочетание срока бронирования и размера когорты может выделять группу со своей долей заездов. Сравним исключение таких записей при обучении с оценкой отдельной доли заездов.

## 6. Статистический и ML-подходы к поиску аномалий

Сравниваем методы из лонгрида «Обнаружение аномалий» (разделы 2.1.1 и 2.2.4). **IQR** отмечает выброс за границами Q1 − 1,5×IQR и Q3 + 1,5×IQR хотя бы по одной координате. Он интерпретируем и устойчив к отдельным крайним значениям. **Isolation Forest** ищет необычные сочетания тех же координат. Его порог задаём по обучению с примерно такой же долей отмеченных броней, как у IQR. Сравниваем пересечение флагов, типичные примеры и результаты прогнозов. Разметки ошибок нет, поэтому согласие методов описывает их поведение, а точность прогнозирования проверяем экспериментом.

Оба метода видят срок бронирования, размер когорты и отклонение срока от дневной медианы. Границы и порог фиксируются по историческому периоду. Isolation Forest подгоняется без целевого признака, чтобы выделить необычные записи. Сравним количество флагов, их пересечение и конкретные примеры. Флаг означает необычное наблюдение и сам по себе не подтверждает ошибку данных.


In [ ]:
scored_snapshots, iqr_limits = {}, {}
anomaly_rows = []
for h, (active, _, _) in snapshots.items():
    scored, limits, _ = detect_anomalies(active, splits[h]['train'], seed=SEED)
    scored_snapshots[h] = scored
    iqr_limits[h] = limits
    for part in ('train', 'validation', 'test'):
        cohort = scored.loc[scored['arrival'].isin(splits[h][part])]
        iqr, forest = cohort['iqr'], cohort['forest']
        anomaly_rows.append({
            'Горизонт': h, 'Период': part, 'Броней': len(cohort),
            'Только IQR': (iqr & ~forest).sum(), 'Только IF': (~iqr & forest).sum(),
            'Оба': (iqr & forest).sum(), 'Ни один': (~iqr & ~forest).sum(),
            'Отмечено IQR, %': 100 * iqr.mean(), 'Отмечено IF, %': 100 * forest.mean(),
            'Индекс Жаккара': (iqr & forest).sum() / max(1, (iqr | forest).sum())
        })
anomalies = pd.DataFrame(anomaly_rows).set_index(['Горизонт', 'Период'])
display(anomalies.round(3))
display(iqr_limits[7].round(1))

inspect7 = scored_snapshots[7].loc[lambda d: d['arrival'].isin(splits[7]['train'])]
same_flags = inspect7['iqr'] & inspect7['forest']
example_cols = ['arrival', 'LeadTime', 'active_count', 'over_median_lead',
                'forest_score', 'nights', 'guests', 'ADR', 'ReservationStatus']
examples = pd.concat({
    'Только IQR': inspect7.loc[inspect7['iqr'] & ~inspect7['forest']].nlargest(4, 'LeadTime'),
    'Только IF': inspect7.loc[inspect7['forest'] & ~inspect7['iqr']].nlargest(4, 'forest_score'),
    'Оба': inspect7.loc[same_flags].nlargest(4, 'forest_score'),
})[example_cols]
display(examples.round({'forest_score': 3, 'ADR': 2}))

fig, ax = plt.subplots(figsize=(8.3, 4.1), layout='constrained')
for name, mask, color in [
    ('Без флага', ~inspect7['iqr'] & ~inspect7['forest'], '#b4bbc1'),
    ('Только IQR', inspect7['iqr'] & ~inspect7['forest'], '#277887'),
    ('Только IF', ~inspect7['iqr'] & inspect7['forest'], '#c26d35'),
    ('Оба метода', same_flags, '#76549b'),
]:
    data = inspect7.loc[mask]
    data = data.sample(n=min(700, len(data)), random_state=SEED)
    ax.scatter(data['LeadTime'], data['active_count'], s=12, alpha=.54, label=name, color=color)
ax.set(title='Какие брони отметили детекторы на горизонте 7 дней',
       xlabel='От бронирования до заезда, дни', ylabel='Активные брони на дату')
ax.legend(frameon=False)
show_figure(fig)


## 7. Совместная плотность и комбинации флагов

По лекции недели 12 (слайды 13, 16, 20, 23–24, 28–30, 34–38) редкость и ошибка данных требуют разной обработки. В дополнение к IQR и Isolation Forest сравниваем kNN, LOF, One-Class SVM, ансамбль рангов и KDE. IQR проверяет отдельные координаты. KDE оценивает совместную плотность трёх признаков и может отметить сочетание, в котором каждая координата лежит внутри своей границы.

- **IQR ≥ 2** отмечает запись при отклонении минимум по двум координатам. Два признака связаны со сроком бронирования, поэтому два срабатывания не считаем независимыми свидетельствами.
- **IQR ИЛИ IF** расширяет охват. **IQR И IF** оставляет совпадения методов.
- **≥ 3 из 4 ML** требует согласия минимум трёх из Isolation Forest, kNN, LOF и One-Class SVM. Ансамбль рангов усредняет процентили их скоров по распределению на train.

Масштабирование RobustScaler, модели и пороги обучаем отдельно для каждого горизонта на train. Для kNN и LOF используем 20 соседей, для One-Class SVM и KDE — фиксированные подвыборки максимум 8 000 и 4 000 броней. KDE использует гауссово ядро с фиксированной шириной 0,25 после масштабирования. Пороги скоринговых методов выбираем по квантилю train с целевой долей флагов IQR. Совпадающие скоры могут уменьшить фактическую долю. У логических правил объём флагов зависит от пересечений.

Разметки подтверждённых ошибок нет. Сверяем пересечения, примеры и долю состоявшихся заездов среди отмеченных. Слайды 42–55 обсуждают отклонения и разладки целого временного ряда. Здесь флаг относится к отдельной брони, а временной контекст учитываем календарными срезами EDA и хронологической валидацией.

In [ ]:
from src.anomaly_comparison import (add_anomaly_detectors, add_combined_flags,
                                    add_mean_flags, make_vote_flags)
from src.forecast import forecast_by_flag

methods = {
    'IQR': 'iqr', 'Isolation Forest': 'forest', 'kNN': 'knn', 'LOF': 'lof',
    'One-Class SVM': 'ocsvm', 'Ансамбль рангов': 'ensemble', 'KDE': 'kde',
    'IQR ≥ 2': 'iqr_two', 'IQR ИЛИ IF': 'iqr_or_forest', 'IQR И IF': 'iqr_and_forest',
    '≥ 3 из 4 ML': 'ml_majority',
}
scored_by_horizon = {}
for h, scored in scored_snapshots.items():
    train = splits[h]['train']
    scored = add_anomaly_detectors(scored, train, seed=SEED)
    scored_by_horizon[h] = add_combined_flags(scored, train, iqr_limits[h], seed=SEED)

profiles = []
for h, scored in scored_by_horizon.items():
    for part, days in splits[h].items():
        cohort = scored.loc[scored['arrival'].isin(days)]
        for method, flag in methods.items():
            flagged = cohort[flag]
            profiles.append({
                'Горизонт': h, 'Период': part, 'Метод': method,
                'Отмечено': int(flagged.sum()), 'Отмечено, %': 100 * flagged.mean(),
                'Заездов под флагом': int(cohort.loc[flagged, 'show'].sum()),
                'Доля заездов под флагом, %': 100 * cohort.loc[flagged, 'show'].mean(),
                'Доля заездов без флага, %': 100 * cohort.loc[~flagged, 'show'].mean(),
            })
flag_profiles = pd.DataFrame(profiles).set_index(['Горизонт', 'Период', 'Метод']).sort_index()
display(flag_profiles.loc[(7, 'train')].round(2))
historical7 = scored_by_horizon[7].loc[lambda d: d['arrival'].isin(splits[7]['train'])]
kde_only = historical7.loc[historical7['kde'] & ~historical7['iqr']]
print(f'KDE отметил {len(kde_only)} исторических броней за 7 дней внутри всех границ IQR.')
display(kde_only.nlargest(5, 'kde_score')[
    ['arrival', 'LeadTime', 'active_count', 'over_median_lead', 'ReservationStatus']
])

## 8. Эксперимент с очисткой истории и учётом флагов

Исторический baseline прогнозирует число активных броней на дату, умноженное на среднюю долю заездов в train. Для каждого флага проверяем две стратегии.

1. **Исключить из истории.** Считаем среднюю долю заездов по немаркированной части train и применяем её ко всем активным броням будущей даты.
2. **Учесть флаг.** Оцениваем две доли заездов на train, для отмеченных и остальных броней. Суммируем вероятности по полной будущей когорте. Для отсутствующей в train группы используем общую долю.

Факт и состав будущей когорты одинаковы для всех вариантов. Сначала фиксируем выбор по среднему MAE на validation, затем считаем результат на test. Тестовые значения остальных заранее заданных стратегий показаны для сравнения. Те же даты test уже были использованы в предыдущей версии проекта, поэтому это ретроспективная проверка.

In [ ]:
BASELINE = 'Без фильтрации'
forecasts = {h: {} for h in HORIZONS}
rows = []
for h, (_, daily, _) in snapshots.items():
    scored, periods = scored_by_horizon[h], splits[h]
    rate, forecasts[h][BASELINE], _ = forecast_baseline(scored, periods['train'], daily['n'])
    histories = {BASELINE: (0, rate)}
    for method, flag in methods.items():
        rate, forecast, excluded = forecast_baseline(scored, periods['train'], daily['n'], flag)
        name = f'Исключить · {method}'
        forecasts[h][name] = forecast
        histories[name] = (excluded, rate)
        name = f'Учесть флаг · {method}'
        forecasts[h][name] = forecast_by_flag(scored, periods['train'], daily.index, flag)
        histories[name] = (0, np.nan)

    validation = periods['validation']
    for name, forecast in forecasts[h].items():
        excluded, rate = histories[name]
        assert forecast.ge(0).all() and forecast.le(daily['n']).all()
        rows.append({
            'Горизонт': h, 'Стратегия': name, 'Исключено из train': excluded,
            'Единая доля заездов, %': 100 * rate,
            'MAE validation': mean_absolute_error(daily.loc[validation, 'actual'], forecast.loc[validation]),
        })

comparison = pd.DataFrame(rows).set_index(['Стратегия', 'Горизонт'])
ranking = comparison['MAE validation'].groupby(level='Стратегия', sort=False).mean().sort_values(kind='stable')
best_option = ranking.index[0]
display(Markdown(f'**Выбор по validation** — {best_option} (средний MAE {ranking.iloc[0]:.4f}).'))

In [ ]:
test_rows = []
for name, h in comparison.index:
    daily = snapshots[h][1]
    test = splits[h]['test']
    actual = daily.loc[test, 'actual']
    prediction = forecasts[h][name].loc[test]
    test_rows.append({
        'MAE test': mean_absolute_error(actual, prediction),
        'Смещение test (прогноз − факт)': (prediction - actual).mean(),
    })
comparison = comparison.join(pd.DataFrame(test_rows, index=comparison.index))
for column in ['MAE validation', 'MAE test']:
    baseline_by_horizon = comparison.loc[BASELINE, column]
    comparison[f'Δ {column} к baseline'] = comparison[column].sub(baseline_by_horizon, level='Горизонт')
summary = comparison.select_dtypes('number').groupby(level='Стратегия').mean().loc[ranking.index]
display(summary[['MAE validation', 'MAE test', 'Δ MAE test к baseline']].round(4))
display(comparison.loc[ranking.index].round(3))

fig, axes = plt.subplots(3, 1, figsize=(11.6, 7.4), sharex=True, layout='constrained')
for ax, h in zip(axes, HORIZONS):
    test = splits[h]['test']
    ax.plot(test, snapshots[h][1].loc[test, 'actual'], lw=1.25, color='#333b45', label='Факт')
    ax.plot(test, forecasts[h][BASELINE].loc[test], lw=1.25, color='#277887', label='Baseline')
    if best_option != BASELINE:
        ax.plot(test, forecasts[h][best_option].loc[test], lw=1.1, color='#c26d35', label='Выбор по validation')
    ax.set(title=f'Горизонт {h} дн.', ylabel='Заезды')
axes[0].legend(frameon=False, ncol=3)
axes[-1].xaxis.set_major_locator(mdates.MonthLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%m.%Y'))
axes[-1].set_xlabel('Дата планового заезда')
show_figure(fig)

## 9. Брони для ручной проверки

В список попадают записи с нулём гостей или нулём ночей. Например, если указан выезд и при этом число гостей равно нулю, запись стоит проверить. Ноль ночей может означать выезд в день заезда. Оба случая требуют проверки по источнику.

В таблице показан тестовый период для броней, активных за 7 дней до заезда. Столбец «One-Class SVM» показывает, отметила ли модель эту же бронь. Номер записи — её порядковый номер в H2 без учёта заголовка. Полный список по всему H2 доступен в `booking_review`. В нём для записей вне когорты указано, что модель их не оценивала.

Дополнительно смотрим, как пересекаются два предметных условия и сколько среди отмеченных записей было заездов. Эти условия служат для аудита исходной выгрузки. Эксперимент с прогнозом использует флаги на признаках, восстановленных к дате среза.

In [ ]:
review_horizon = 7
review_scores = scored_by_horizon[review_horizon]
zero_guests = bookings['guests'].eq(0)
zero_nights = bookings['nights'].eq(0)
review_rows = bookings.loc[zero_guests | zero_nights]
review_reason = np.select(
    [review_rows['guests'].eq(0) & review_rows['nights'].eq(0), review_rows['guests'].eq(0)],
    ['Ноль гостей и ночей', 'Ноль гостей'],
    default='Ноль ночей',
)
model_flags = review_scores['ocsvm'].map({True: 'Отметила', False: 'Пропустила'})
booking_review = pd.DataFrame({
    'Запись H2': review_rows.index + 1,
    'Дата заезда': review_rows['arrival'],
    'Гостей': review_rows['guests'].astype('Int64'),
    'Ночей': review_rows['nights'],
    'Статус': review_rows['ReservationStatus'].map({
        'Check-Out': 'Выехали', 'Canceled': 'Отменена', 'No-Show': 'Неявка',
    }),
    'Причина проверки': review_reason,
    'One-Class SVM': model_flags.reindex(review_rows.index, fill_value='Вне когорты'),
}).sort_values(['Дата заезда', 'Запись H2'])

test_dates = splits[review_horizon]['test']
test_booking_ids = review_scores.index[review_scores['arrival'].isin(test_dates)]
test_review = booking_review.loc[booking_review.index.isin(test_booking_ids)]
flagged_count = int(test_review['One-Class SVM'].eq('Отметила').sum())
display(Markdown(
    f"Во всём H2 записей для проверки — **{len(booking_review)}**. "
    f"В тестовой когорте — **{len(test_review)}**. "
    f"Из них модель отметила **{flagged_count}**, пропустила **{len(test_review) - flagged_count}**."
))
display(test_review.reset_index(drop=True))

audit_conditions = {
    'Ноль гостей': zero_guests,
    'Ноль ночей': zero_nights,
    'Ноль гостей ИЛИ ноль ночей': zero_guests | zero_nights,
    'Ноль гостей И ноль ночей': zero_guests & zero_nights,
}
display(pd.DataFrame.from_dict({
    name: {'Строк в H2': int(mask.sum()), 'Из них заезды': int(bookings.loc[mask, 'show'].sum())}
    for name, mask in audit_conditions.items()
}, orient='index'))

## 10. LOF, kNN, DBSCAN и правила по средним

На тех же трёх признаках kNN измеряет среднее расстояние до соседей, LOF сравнивает плотность с плотностью соседей, а DBSCAN ищет плотные области фиксированного радиуса. Для DBSCAN берём до 8 000 исторических броней в масштабе RobustScaler. Радиус задаём по распределению расстояний до 20-го соседа без учёта самой точки, по квантилю, соответствующему доле флагов IQR. В DBSCAN это 21 точка с учётом самой брони. На будущих данных отмечаем точки вне радиуса любой исторической опорной точки DBSCAN. Так будущие записи оцениваются независимо друг от друга. Доля флагов может отличаться от IQR. [DBSCAN](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html) · [LOF на новых наблюдениях](https://scikit-learn.org/stable/auto_examples/neighbors/plot_lof_novelty_detection.html)

Добавили два статистических правила с порогом три стандартных отклонения. Первое использует общее среднее train по каждому признаку. Второе использует среднее и стандартное отклонение внутри дня недели планового заезда. Так, объём пятничной когорты сравнивается с историческими пятницами. День недели известен на дату прогноза. В H2 нет идентификатора клиента для построения личной нормы. Для группы с менее чем 30 бронями или нулевым разбросом по признаку используем общую статистику. Статистики взвешены по броням. Три стандартных отклонения здесь задают эвристику, а фактический охват измеряем на данных.

Перебираем все 36 пар и 84 тройки из девяти самостоятельных детекторов. Для пар берём «хотя бы один» и «оба». Для троек проверяем «хотя бы один», «два из трёх» и «все три». Всего 324 правила. Голоса близких методов могут совпадать. Пересечение оцениваем индексом Жаккара, долей общих флагов среди всех броней, отмеченных хотя бы одним из сравниваемых методов.

In [ ]:
individual = {
    'IQR': 'iqr', 'Isolation Forest': 'forest', 'kNN': 'knn', 'LOF': 'lof',
    'One-Class SVM': 'ocsvm', 'KDE': 'kde', 'DBSCAN': 'dbscan',
    'Среднее общее': 'mean_global', 'Среднее по дню недели': 'mean_weekday',
}
extended_scored = {h: add_mean_flags(scored, splits[h]['train']) for h, scored in scored_by_horizon.items()}
train7_extended = extended_scored[7].loc[lambda d: d['arrival'].isin(splits[7]['train'])]
train_flags7 = train7_extended[list(individual.values())].rename(columns={v: k for k, v in individual.items()})
display(pd.DataFrame({
    'Броней под флагом': train_flags7.sum(),
    'Под флагом, %': 100 * train_flags7.mean(),
    'Заездов под флагом': train_flags7.loc[train7_extended['show'].eq(1)].sum(),
}).round(2))

intersections = train_flags7.astype('int32').T @ train_flags7.astype('int32')
counts = train_flags7.sum().to_numpy()
unions = counts[:, None] + counts[None, :] - intersections
display(Markdown('**Жаккар по флагам на train, 7 дней**'))
display(intersections.div(unions.where(unions.gt(0))).round(2))

weekday_examples = train7_extended.loc[
    train7_extended['mean_weekday'] & ~train7_extended['mean_global'],
    ['arrival', 'LeadTime', 'active_count', 'over_median_lead', 'ReservationStatus'],
].copy()
weekday_examples['День недели'] = weekday_examples['arrival'].dt.day_of_week.map(dict(enumerate(weekdays)))
print(f'У {len(weekday_examples)} броней флаг по дню недели при отсутствии общего флага.')
display(weekday_examples.head(6))

### Эксперимент на дневном прогнозе

Для каждого правила и одиночного детектора проверяем удаление из истории и расчёт отдельной вероятности для отмеченных. Те же даты, MAE и полный состав прогнозируемой когорты используются во всех сравнениях. Сначала ранжируем по validation. Для каждой стратегии считаем результат по всем трём горизонтам. Если правило удалило всю историю на одном из горизонтов, стратегия удаления исключается из общего рейтинга. При отсутствии одной из групп на train её вероятность равна общей исторической доле.

Результаты на test служат ретроспективной проверкой. Этот период уже использовался раньше, а перебор сотен вариантов увеличивает шанс случайного выигрыша на validation. В конце показываем лидеров каждой группы. Полная сводка и результаты по горизонтам доступны в `voting_summary` и `voting_by_horizon`.

In [ ]:
voting_predictions, voting_metadata = {}, {}
for h, scored in extended_scored.items():
    daily = snapshots[h][1]
    individual_flags = scored[list(individual.values())].set_axis(list(individual), axis=1)
    votes = make_vote_flags(scored, individual)
    assert votes.shape[1] == 324
    flags = pd.concat([individual_flags, votes], axis=1)
    train_mask = scored['arrival'].isin(splits[h]['train'])
    train_size = int(train_mask.sum())
    total_shows = int(scored.loc[train_mask, 'show'].sum())
    flagged_train = flags.loc[train_mask].sum()
    flagged_shows = flags.loc[train_mask & scored['show'].eq(1)].sum()
    overall = total_shows / train_size
    clean_rate = (total_shows - flagged_shows) / (train_size - flagged_train).replace(0, np.nan)
    flag_rate = flagged_shows / flagged_train.replace(0, np.nan)
    flagged_daily = flags.groupby(scored['arrival']).sum().reindex(daily.index, fill_value=0)
    removed = pd.DataFrame(np.outer(daily['n'], clean_rate), index=daily.index, columns=flags.columns)
    segmented = (
        flagged_daily.mul(flag_rate.fillna(overall))
        + flagged_daily.rsub(daily['n'], axis=0).mul(clean_rate.fillna(overall))
    )
    voting_predictions[h] = {
        'Исключить': removed,
        'Учесть флаг': segmented,
        'Baseline': forecasts[h][BASELINE].to_frame(BASELINE),
    }
    voting_metadata[h] = pd.DataFrame({
        'Под флагом на train, %': 100 * flagged_train / train_size,
        'Заездов под флагом на train, %': 100 * flag_rate,
    })

def voting_mae(part):
    tables = {}
    for h, policies in voting_predictions.items():
        dates = splits[h][part]
        actual = snapshots[h][1].loc[dates, 'actual']
        tables[h] = pd.concat({
            policy: predicted.loc[dates].sub(actual, axis=0).abs().mean()
            for policy, predicted in policies.items()
        }, names=['Стратегия', 'Правило'])
    return pd.concat(tables, names=['Горизонт']).rename(f'MAE {part}')

voting_validation = voting_mae('validation')
voting_ranking = voting_validation.dropna().groupby(level=['Стратегия', 'Правило'], sort=False).agg(['mean', 'size'])
voting_ranking = voting_ranking.loc[voting_ranking['size'].eq(len(HORIZONS)), 'mean'].sort_values(kind='stable')
voting_choice = voting_ranking.index[0]
display(Markdown(f'**Выбор по validation** — {voting_choice[0]} · {voting_choice[1]}.'))

In [ ]:
voting_by_horizon = pd.concat([voting_validation, voting_mae('test')], axis=1)
voting_by_horizon = voting_by_horizon.join(
    pd.concat(voting_metadata, names=['Горизонт', 'Правило']), on=['Горизонт', 'Правило']
)
voting_summary = voting_by_horizon.groupby(level=['Стратегия', 'Правило'], sort=False).mean().loc[voting_ranking.index]
voting_summary['Δ MAE test к baseline'] = voting_summary['MAE test'] - voting_summary.loc[('Baseline', BASELINE), 'MAE test']
rule_names = voting_summary.index.get_level_values('Правило')
voting_summary['Размер набора'] = np.select(
    [rule_names == BASELINE, rule_names.isin(individual), rule_names.str.contains(' из 2 · ', regex=False)],
    [0, 1, 2], default=3,
)
leaders = voting_summary.groupby('Размер набора', sort=True).head(3)
display(leaders.round(4))
display(Markdown('**Отдельные детекторы**'))
display(voting_summary.loc[voting_summary['Размер набора'].eq(1)].round(4))

nearby_triple = rule_names.str.endswith('kNN + LOF + DBSCAN')
display(Markdown('**Три порога голосования для kNN, LOF и DBSCAN**'))
display(voting_summary.loc[nearby_triple].round(4))
display(Markdown('**Лучшая очистка по validation**'))
display(voting_summary.xs('Исключить', level='Стратегия').head(1).round(4))

## Короткий отчёт для LMS

**Данные и EDA.** H2 содержит 79 330 броней и 31 столбец по городскому отелю. Медиана срока бронирования — 74 дня. В полной истории доля заездов среди активных броней растёт с 88,2% на горизонте 14 дней до 96,2% за 3 дня. На train за 7 дней до заезда доля составляет 85,0% для броней со сроком 7–13 дней и 94,6% для 120+ дней. Медиана размера дневной когорты — 35 броней во вторник и 62 в пятницу. Эти различия мотивируют проверку срока бронирования и календаря в модели.

**Baseline и валидация.** Прогноз — число активных броней, умноженное на историческую долю заездов. Разбили данные по плановым датам и отфильтровали даты с исходами, которые на момент следующего окна ещё не были известны. Стратегию выбирали по среднему MAE на validation. На test 123 даты заезда.

| Горизонт | MAE baseline | MAE kNN | MAE IQR ИЛИ kNN ИЛИ LOF |
| --- | ---: | ---: | ---: |
| 14 дней | 4,344 | 3,666 | 3,906 |
| 7 дней | 3,891 | 3,473 | 3,478 |
| 3 дня | 1,821 | 1,727 | 1,656 |
| Среднее | 3,352 | 2,955 | 3,013 |

**Что отметили детекторы.** На train для горизонта 7 дней IQR и Isolation Forest вместе отметили 3 368 броней, а правило «ИЛИ» — 5 274. Из броней под обоими флагами 3 147 завершились заездом. KDE выделил ещё 1 592 брони внутри всех одномерных границ IQR. В исходной выгрузке у 431 записи ноль гостей или ночей, из них у 67 выполнены оба условия. Это кандидаты на ручную проверку по данным отеля. Разметки подтверждённых ошибок в H2 нет.

**Сравнение по средним и плотности.** На train за 7 дней общее правило 3σ отметило 1 135 броней, правило по дню недели — 988. Из них 70 отметил только групповой метод. Например, для брони на воскресенье 11.09.2016 срок превышал медиану своей когорты на 207 дней. Максимальный |z| вырос с 2,99 по всей истории до 3,23 относительно воскресений. Заезд состоялся. kNN, LOF и DBSCAN отметили 4 320, 4 321 и 2 625 броней соответственно. У LOF и DBSCAN Жаккар составил 0,13. При отдельной вероятности заезда общий и групповой флаги дали тестовый MAE 3,272 и 3,275 в среднем по горизонтам.

**Результат и решение.** Среди 36 пар по validation лидировало правило «IQR ИЛИ LOF» с отдельной долей заездов для отмеченных (MAE 1,497). Среди 84 троек и в общем рейтинге лидировало «IQR ИЛИ kNN ИЛИ LOF» (1,489 против 1,515 у одиночного kNN и 1,549 у baseline). На тесте тройка улучшила baseline с 3,352 до 3,013. Одиночный kNN дал 2,955. Лучшая очистка по validation, «kNN И One-Class SVM», дала на тесте 3,324. Сохраняем все брони. Перед выбором между одиночным kNN и тройкой проверим их на нескольких новых временных окнах. Оба текущих результата относятся к уже использованному летнему периоду H2.

## Первый эксперимент этапа 3

Сравниваем логистическую регрессию на отдельных бронях с историческим baseline и стратегией kNN из первого эксперимента этапа 2. Модель использует логарифм LeadTime, размер когорты, отклонение LeadTime от медианы по когорте и циклические признаки дня недели и месяца планового заезда. Суммируем вероятности заезда до дневного прогноза. Во второй конфигурации добавляем фиксированный флаг **IQR И IF** и сравниваем результаты на тех же датах. Обе конфигурации сохраняют все брони. Признаки масштабируются на train, остальные параметры модели фиксированы.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

logistic_predictions = {}
for h, scored in scored_by_horizon.items():
    arrival = scored['arrival'].dt
    features = scored[['active_count', 'over_median_lead']].assign(log_lead=np.log1p(scored['LeadTime']))
    for name, values, period in [('weekday', arrival.day_of_week, 7), ('month', arrival.month - 1, 12)]:
        phase = 2 * np.pi * values / period
        features[f'{name}_sin'] = np.sin(phase)
        features[f'{name}_cos'] = np.cos(phase)
    train_mask = scored['arrival'].isin(splits[h]['train'])
    logistic_predictions[h] = {name: forecasts[h][name] for name in dict.fromkeys([BASELINE, best_option])}
    for name, inputs in {
        'Логистическая регрессия': features,
        'Логистическая регрессия + флаг': features.assign(anomaly=scored['iqr_and_forest'].astype(int)),
    }.items():
        model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500))
        model.fit(inputs.loc[train_mask], scored.loc[train_mask, 'show'])
        probabilities = pd.Series(model.predict_proba(inputs)[:, 1], index=inputs.index)
        daily = snapshots[h][1]
        logistic_predictions[h][name] = probabilities.groupby(scored['arrival']).sum().reindex(daily.index, fill_value=0.0)

def stage3_mae(part):
    rows = []
    for h, predictions in logistic_predictions.items():
        dates = splits[h][part]
        actual = snapshots[h][1].loc[dates, 'actual']
        rows.append(pd.Series({name: mean_absolute_error(actual, forecast.loc[dates])
                               for name, forecast in predictions.items()}, name=h))
    return pd.DataFrame(rows).rename_axis('Горизонт')

stage3_validation = stage3_mae('validation')
stage3_choice = stage3_validation.mean().idxmin()
stage3_test = stage3_mae('test')
display(pd.concat({'MAE validation': stage3_validation, 'MAE test': stage3_test}, axis=1).round(3))
display(Markdown(f'**Выбор по validation** — {stage3_choice}.'))

Логистическая регрессия с флагом немного улучшила результат той же модели без флага на всех горизонтах. Средний MAE модели с флагом — 1,929 на validation и 3,009 на test. В сравнении этих четырёх конфигураций стратегия с отдельной долей для флага kNN сохранила лучший MAE на validation и test. Следующий эксперимент — сравнить подходы на нескольких временных срезах.